In [64]:
import pandas as pd

In [65]:
merged= pd.read_csv("merged_final.csv")

In [66]:
cols = ['booking_id', 'slot_at', 'city', 'channel',
        'patient_id', 'test_code', 'test_price', 'home_fee']

lines = merged[cols]

print(lines.shape)
lines.head()

(1311662, 8)


,booking_id,slot_at,city,channel,patient_id,test_code,test_price,home_fee
0,BK-AHM-202501-000001,2025-01-09 08:26:00,Ahmedabad,Clinic,PAHM0028525,T009,800.0,0
1,BK-AHM-202501-000001,2025-01-09 08:26:00,Ahmedabad,Clinic,PAHM0028525,T004,650.0,0
2,BK-AHM-202501-000001,2025-01-09 08:26:00,Ahmedabad,Clinic,PAHM0028525,T007,550.0,0
3,BK-AHM-202501-000002,2025-01-06 13:22:00,Ahmedabad,Clinic,PAHM0031384,T004,650.0,0
4,BK-AHM-202501-000003,2025-01-28 13:37:00,Ahmedabad,Home,PAHM0027001,T003,450.0,150


In [67]:
lines['is_test'] = lines['test_price'].notna()
print('rows:', len(lines))
print('real tests:', lines['is_test'].sum())

rows: 1311662
real tests: 1311642


In [68]:
bookings = lines.groupby('booking_id').agg({
    'slot_at': 'first',
    'city': 'first',
    'channel': 'first',
    'patient_id': 'first',
    'home_fee': 'first',
    'is_test': 'sum'
}).reset_index()

bookings = bookings.rename(columns={'is_test': 'n_tests'})
print('bookings:', len(bookings))

bookings: 566507


In [69]:
bookings['visit'] = pd.to_datetime(bookings['slot_at'])
bookings['quarter'] = bookings['visit'].dt.to_period('Q').astype(str)
bookings['month'] = bookings['visit'].dt.strftime('%Y-%m')
bookings['year'] = bookings['visit'].dt.year
bookings['first_half'] = bookings['visit'].dt.month <= 6
bookings['channel'] = bookings['channel'].str.lower()
bookings.head()

,booking_id,slot_at,city,channel,patient_id,home_fee,n_tests,visit,quarter,month,year,first_half
0,BK-AHM-202501-000001,2025-01-09 08:26:00,Ahmedabad,clinic,PAHM0028525,0,3,2025-01-09 08:26:00,2025Q1,2025-01,2025,True
1,BK-AHM-202501-000002,2025-01-06 13:22:00,Ahmedabad,clinic,PAHM0031384,0,1,2025-01-06 13:22:00,2025Q1,2025-01,2025,True
2,BK-AHM-202501-000003,2025-01-28 13:37:00,Ahmedabad,home,PAHM0027001,150,4,2025-01-28 13:37:00,2025Q1,2025-01,2025,True
3,BK-AHM-202501-000004,2025-01-24 07:15:00,Ahmedabad,clinic,PAHM0007069,0,1,2025-01-24 07:15:00,2025Q1,2025-01,2025,True
4,BK-AHM-202501-000005,2025-01-11 09:56:00,Ahmedabad,clinic,PAHM0002520,0,2,2025-01-11 09:56:00,2025Q1,2025-01,2025,True


In [70]:
bookings['in_offer'] = (bookings['channel'] == 'home') & (bookings['home_fee'] == 0) & (bookings['quarter'] == '2026Q1')
print('offer bookings:', bookings['in_offer'].sum())

offer bookings: 48869


In [71]:
patients = pd.read_csv("cleaned_patients.csv", usecols=['patient_id', 'signup_date'])
patients['signup_quarter'] = pd.to_datetime(patients['signup_date']).dt.to_period('Q').astype(str)
print('patients:', len(patients))

patients: 303153


In [72]:
bookings = bookings.merge(patients[['patient_id', 'signup_quarter']], on='patient_id', how='left')

bookings['patient_type'] = 'returning'
bookings.loc[bookings['signup_quarter'] == bookings['quarter'], 'patient_type'] = 'new'
bookings['patient_type'].value_counts()

patient_type
returning    405530
new          160977
Name: count, dtype: int64

In [73]:
bookings

,booking_id,slot_at,city,channel,patient_id,home_fee,n_tests,visit,quarter,month,year,first_half,in_offer,signup_quarter,patient_type
0,BK-AHM-202501-000001,2025-01-09 08:26:00,Ahmedabad,clinic,PAHM0028525,0,3,2025-01-09 08:26:00,2025Q1,2025-01,2025,True,False,2024Q1,returning
1,BK-AHM-202501-000002,2025-01-06 13:22:00,Ahmedabad,clinic,PAHM0031384,0,1,2025-01-06 13:22:00,2025Q1,2025-01,2025,True,False,2024Q2,returning
2,BK-AHM-202501-000003,2025-01-28 13:37:00,Ahmedabad,home,PAHM0027001,150,4,2025-01-28 13:37:00,2025Q1,2025-01,2025,True,False,2023Q4,returning
3,BK-AHM-202501-000004,2025-01-24 07:15:00,Ahmedabad,clinic,PAHM0007069,0,1,2025-01-24 07:15:00,2025Q1,2025-01,2025,True,False,2024Q4,returning
4,BK-AHM-202501-000005,2025-01-11 09:56:00,Ahmedabad,clinic,PAHM0002520,0,2,2025-01-11 09:56:00,2025Q1,2025-01,2025,True,False,2024Q4,returning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566502,nova_jai_0055749,2026-06-08 11:54:00,Jaipur,home,PJAI0028824,150,2,2026-06-08 11:54:00,2026Q2,2026-06,2026,True,False,2026Q2,new
566503,nova_jai_0055750,2026-06-18 08:22:00,Jaipur,home,PJAI0028764,150,3,2026-06-18 08:22:00,2026Q2,2026-06,2026,True,False,2026Q2,new
566504,nova_jai_0055751,2026-06-26 11:47:00,Jaipur,clinic,PJAI0028485,0,1,2026-06-26 11:47:00,2026Q2,2026-06,2026,True,False,2026Q2,new
566505,nova_jai_0055752,2026-06-12 11:18:00,Jaipur,home,PJAI0017296,150,3,2026-06-12 11:18:00,2026Q2,2026-06,2026,True,False,2024Q4,returning


Boards 5%

In [74]:
tests_2025 = bookings[(bookings['year'] == 2025) & bookings['first_half']]['n_tests'].sum()
tests_2026 = bookings[(bookings['year'] == 2026) & bookings['first_half']]['n_tests'].sum()
growth = (tests_2026 / tests_2025 - 1) * 100

print('Jan-Jun 2025 tests:', tests_2025)
print('Jan-Jun 2026 tests:', tests_2026)
print('growth:', round(growth, 2), '%')

Jan-Jun 2025 tests: 436302
Jan-Jun 2026 tests: 457982
growth: 4.97 %


Who got the offer?

In [75]:
q1 = bookings[bookings['quarter'] == '2026Q1']
offer_users = q1[q1['in_offer']]['patient_id'].unique()

q1_patients = pd.DataFrame({'patient_id': q1['patient_id'].unique()})
q1_patients['used_offer'] = q1_patients['patient_id'].isin(offer_users)
q1_patients['used_offer'].value_counts()

used_offer
True     37320
False    25644
Name: count, dtype: int64

In [76]:
bookings_2025 = bookings[bookings['year'] == 2025]

count_2025 = bookings_2025.groupby('patient_id')['booking_id'].count().reset_index()
count_2025.columns = ['patient_id', 'bookings_2025']

home_2025 = bookings_2025[bookings_2025['channel'] == 'home']['patient_id'].unique()
clinic_2025 = bookings_2025[bookings_2025['channel'] == 'clinic']['patient_id'].unique()

In [77]:
q1_patients = q1_patients.merge(count_2025, on='patient_id', how='left')
q1_patients['bookings_2025'] = q1_patients['bookings_2025'].fillna(0)
q1_patients['used_home_2025'] = q1_patients['patient_id'].isin(home_2025)

summary = q1_patients.groupby('used_offer').agg({'bookings_2025': 'mean', 'used_home_2025': 'mean'})
summary['bookings_per_quarter_2025'] = summary['bookings_2025'] / 4
summary['pct_used_home_2025'] = summary['used_home_2025'] * 100
summary[['bookings_per_quarter_2025', 'pct_used_home_2025']].round(3)

,bookings_per_quarter_2025,pct_used_home_2025
used_offer,,
False,0.538,38.613
True,0.673,42.899


In [78]:
offer_bookings = bookings[bookings['in_offer']].copy()

offer_bookings['who'] = 'returning, no 2025 booking'
offer_bookings.loc[offer_bookings['patient_id'].isin(clinic_2025), 'who'] = 'returning, clinic only in 2025'
offer_bookings.loc[offer_bookings['patient_id'].isin(home_2025), 'who'] = 'returning, used home in 2025'
offer_bookings.loc[offer_bookings['signup_quarter'] == '2026Q1', 'who'] = 'new in Q1 2026'

who = offer_bookings['who'].value_counts()
print(who)

who
returning, used home in 2025      22092
new in Q1 2026                    14111
returning, clinic only in 2025    10915
returning, no 2025 booking         1751
Name: count, dtype: int64


In [79]:
existing = len(offer_bookings) - who['new in Q1 2026']
print('from existing patients:', existing, '=', round(existing / len(offer_bookings) * 100), '%')
print('from people who already used home collection:', round(who['returning, used home in 2025'] / len(offer_bookings) * 100), '%')

from existing patients: 34758 = 71 %
from people who already used home collection: 45 %


what would Q1 2026 have looked like without the offer?

In [80]:
table = bookings.groupby(['quarter', 'channel'])['booking_id'].count().unstack()
table

channel,clinic,home
quarter,,
2025Q1,63306,32816
2025Q2,61864,31891
2025Q3,60538,31330
2025Q4,59182,30492
2026Q1,50752,48869
2026Q2,57038,38385
2026Q3,27,17


In [81]:
actual = table.loc['2026Q1'].sum()

a1 = actual - table.loc['2025Q4'].sum()   # previous quarter
a2 = actual - table.loc['2025Q1'].sum()   # same quarter last year
a4 = actual - table.loc['2026Q2'].sum()   # the quarter after the offer

A3: Q2 2026, scaled up by how much bigger Q1 was than Q2 in 2025, for home and clinic separately.

In [82]:
home_base = table.loc['2026Q2', 'home'] * table.loc['2025Q1', 'home'] / table.loc['2025Q2', 'home']
clinic_base = table.loc['2026Q2', 'clinic'] * table.loc['2025Q1', 'clinic'] / table.loc['2025Q2', 'clinic']
a3 = actual - (home_base + clinic_base)

A5: a straight line through the 2025 months, carried on into Jan–Mar 2026.

In [83]:
import numpy as np
monthly_2025 = bookings_2025.groupby(['month', 'channel'])['booking_id'].count().unstack()
x = np.arange(12)

trend_total = 0
for ch in ['home', 'clinic']:
    slope, start = np.polyfit(x, monthly_2025[ch], 1)
    for m in [12, 13, 14]:
        trend_total = trend_total + start + slope * m

a5 = actual - trend_total

In [84]:
a1

np.int64(9947)

In [85]:
a2

np.int64(3499)

In [86]:
a3

np.float64(1755.1313873800973)

In [87]:
a4

np.int64(4198)

In [88]:
a5

np.float64(11840.78671328671)

In [89]:
yardsticks = pd.DataFrame({
    'yardstick': ['A1 previous quarter', 'A2 same quarter last year', 'A3 Q2 2026 with 2025 season',
                  'A4 Q2 2026 as it is', 'A5 2025 trend'],
    'extra_bookings': [round(a1), round(a2), round(a3), round(a4), round(a5)]
})
yardsticks

,yardstick,extra_bookings
0,A1 previous quarter,9947
1,A2 same quarter last year,3499
2,A3 Q2 2026 with 2025 season,1755
3,A4 Q2 2026 as it is,4198
4,A5 2025 trend,11841


split using city

In [90]:
segments = []
for city in sorted(bookings['city'].unique()):
    segments.append(('city', city))
for t in ['returning', 'new']:
    segments.append(('patient_type', t))
for ch in ['home', 'clinic']:
    segments.append(('channel', ch))

In [91]:
results = []
for column, value in segments:
    part = bookings[bookings[column] == value]
    counts = part['quarter'].value_counts()
    results.append({
        'segment': value,
        'Q1 2025': counts['2025Q1'],
        'Q4 2025': counts['2025Q4'],
        'Q1 2026': counts['2026Q1'],
        'vs Q4 2025 %': round((counts['2026Q1'] / counts['2025Q4'] - 1) * 100, 2),
        'vs Q1 2025 %': round((counts['2026Q1'] / counts['2025Q1'] - 1) * 100, 2),
    })

pd.DataFrame(results)

,segment,Q1 2025,Q4 2025,Q1 2026,vs Q4 2025 %,vs Q1 2025 %
0,Ahmedabad,16500,15935,18328,15.02,11.08
1,Bengaluru,22307,19706,23417,18.83,4.98
2,Chennai,17471,17356,17646,1.67,1.00
3,Hyderabad,16127,14838,17053,14.93,5.74
4,Jaipur,10442,9268,8792,-5.14,-15.80
5,Pune,13275,12571,14385,14.43,8.36
6,returning,69015,64409,70805,9.93,2.59
7,new,27107,25265,28816,14.06,6.30
8,home,32816,30492,48869,60.27,48.92
9,clinic,63306,59182,50752,-14.24,-19.83


In [92]:
for column in ['city', 'patient_type']:
    print('---', column)
    for value in sorted(bookings[column].unique()):
        rates = []
        for q in ['2025Q4', '2026Q1']:
            part = bookings[(bookings[column] == value) & (bookings['quarter'] == q)]
            rates.append(len(part) / part['patient_id'].nunique())
        print(value, round(rates[0], 3), '->', round(rates[1], 3))

--- city
Ahmedabad 1.543 -> 1.578
Bengaluru 1.523 -> 1.575
Chennai 1.553 -> 1.611
Hyderabad 1.552 -> 1.577
Jaipur 1.536 -> 1.587
Pune 1.542 -> 1.569
--- patient_type
new 1.409 -> 1.437
returning 1.6 -> 1.65


In [93]:
def make_grid(before_q, after_q):
    after = bookings[bookings['quarter'] == after_q]
    before = bookings[bookings['quarter'] == before_q]

    # one row per patient who booked in the after quarter
    people = after.groupby('patient_id').agg({'city': 'first', 'signup_quarter': 'first', 'booking_id': 'count'})
    people = people.rename(columns={'booking_id': 'after'})
    people['before'] = before.groupby('patient_id')['booking_id'].count()
    people['before'] = people['before'].fillna(0)

    home_patients = after[after['channel'] == 'home']['patient_id'].unique()
    people['group'] = 'no offer'
    people.loc[people.index.isin(home_patients), 'group'] = 'offer'

    people['type'] = 'returning'
    people.loc[people['signup_quarter'] == after_q, 'type'] = 'new'
    return people

In [94]:
def lift(people):
    offer = people[people['group'] == 'offer']
    other = people[people['group'] == 'no offer']

    offer_change = (offer['after'].sum() - offer['before'].sum()) / len(offer)
    other_change = (other['after'].sum() - other['before'].sum()) / len(other)
    extra = (offer_change - other_change) * len(offer)

    if offer['before'].sum() > 0 and other['before'].sum() > 0:
        ratio = (offer['after'].sum() / offer['before'].sum()) / (other['after'].sum() / other['before'].sum()) - 1
        ratio = round(ratio * 100, 1)
    else:
        ratio = None   # new patients have no "before" quarter

    return {'offer patients': len(offer), 'no-offer patients': len(other),
            'offer before': round(offer['before'].mean(), 3), 'offer after': round(offer['after'].mean(), 3),
            'no-offer before': round(other['before'].mean(), 3), 'no-offer after': round(other['after'].mean(), 3),
            'lift %': ratio, 'extra bookings': round(extra)}

In [95]:
people = make_grid('2025Q4', '2026Q1')

grid = []
row = lift(people)
row['segment'] = 'all patients'
grid.append(row)

for city in sorted(people['city'].unique()):
    row = lift(people[people['city'] == city])
    row['segment'] = city
    grid.append(row)

for t in ['returning', 'new']:
    row = lift(people[people['type'] == t])
    row['segment'] = t
    grid.append(row)

pd.DataFrame(grid)

,offer patients,no-offer patients,offer before,offer after,no-offer before,no-offer after,lift %,extra bookings,segment
0,37320,25644,0.885,1.858,0.591,1.182,5.0,14264,all patients
1,6826,4787,0.852,1.859,0.581,1.177,7.6,2802,Ahmedabad
2,8803,6069,0.827,1.849,0.541,1.176,2.8,3408,Bengaluru
3,6518,4438,1.002,1.896,0.662,1.192,5.1,2375,Chennai
4,6405,4410,0.852,1.849,0.551,1.181,1.4,2357,Hyderabad
5,3275,2264,0.995,1.855,0.698,1.200,8.5,1173,Jaipur
6,5493,3676,0.853,1.834,0.585,1.173,7.1,2153,Pune
7,25901,17009,1.275,1.942,0.891,1.206,12.5,9102,returning
8,11419,8635,0.000,1.667,0.000,1.133,NaN,6101,new


trying to break our result

In [96]:
monthly = bookings[bookings['quarter'] != '2026Q3'].groupby(['month', 'channel'])['booking_id'].count().unstack()
monthly

channel,clinic,home
month,,
2025-01,21080,11015
2025-02,21004,10710
2025-03,21222,11091
2025-04,20683,10796
2025-05,20757,10590
2025-06,20424,10505
2025-07,20308,10563
2025-08,19735,9946
2025-09,20495,10821


In [97]:
old_cities = ['Ahmedabad', 'Bengaluru', 'Hyderabad', 'Pune']
four = bookings[bookings['city'].isin(old_cities)]
four_table = four.groupby(['quarter', 'channel'])['booking_id'].count().unstack()
four_table

channel,clinic,home
quarter,,
2025Q1,44973,23236
2025Q2,44147,22936
2025Q3,42899,22230
2025Q4,41697,21353
2026Q1,37225,35958
2026Q2,42047,28377


In [98]:
for q in ['2026Q1', '2026Q2']:
    last_year = q.replace('2026', '2025')
    home_change = (four_table.loc[q, 'home'] / four_table.loc[last_year, 'home'] - 1) * 100
    total_change = (four_table.loc[q].sum() / four_table.loc[last_year].sum() - 1) * 100
    print(q, ' home vs last year:', round(home_change, 1), '%   total vs last year:', round(total_change, 1), '%')

2026Q1  home vs last year: 54.8 %   total vs last year: 7.3 %
2026Q2  home vs last year: 23.7 %   total vs last year: 5.0 %


In [99]:
per_quarter = bookings.groupby(['patient_id', 'quarter'])['booking_id'].count().unstack().fillna(0)
returning = people[people['type'] == 'returning']

for g in ['offer', 'no offer']:
    ids = returning[returning['group'] == g].index
    q4 = per_quarter.loc[ids, '2025Q4'].mean()
    q2 = per_quarter.loc[ids, '2026Q2'].mean()
    print(g, '  Q4 2025:', round(q4, 3), '  Q2 2026:', round(q2, 3), '  change:', round((q2 / q4 - 1) * 100, 1), '%')

offer   Q4 2025: 1.275   Q2 2026: 0.785   change: -38.4 %
no offer   Q4 2025: 0.891   Q2 2026: 0.485   change: -45.6 %


In [100]:
# Calculate total test price for each booking
booking_values = lines.groupby('booking_id')['test_price'].sum()

# Add it to the bookings DataFrame
bookings['test_value'] = bookings['booking_id'].map(booking_values)

# Filter Q1 2026
q1_bookings = bookings[bookings['quarter'] == '2026Q1']

value = q1_bookings['test_value'].sum() / len(q1_bookings)

print('Test value booked in Q1 2026:',
      round(q1_bookings['test_value'].sum()))

print('Value per booking:', round(value))

Test value booked in Q1 2026: 161315382
Value per booking: 1619


In [101]:
q1_bookings = bookings[bookings['quarter'] == '2026Q1']

value = q1_bookings['test_value'].sum() / len(q1_bookings)
print('test value booked in Q1 2026:', round(q1_bookings['test_value'].sum()))
print('value per booking:', round(value))

test value booked in Q1 2026: 161315382
value per booking: 1619


In [102]:
placebo_extras = []
for before_q, after_q in [('2025Q1', '2025Q2'), ('2025Q2', '2025Q3'), ('2025Q3', '2025Q4')]:
    r = lift(make_grid(before_q, after_q))
    placebo_extras.append(r['extra bookings'])
    print(before_q, '->', after_q, '  extra bookings:', r['extra bookings'], '  lift:', r['lift %'], '%')

2025Q1 -> 2025Q2   extra bookings: 9068   lift: 3.4 %
2025Q2 -> 2025Q3   extra bookings: 9816   lift: 7.5 %
2025Q3 -> 2025Q4   extra bookings: 9381   lift: 6.9 %


In [103]:
grid_extra = lift(people)['extra bookings']
placebo_average = sum(placebo_extras) / len(placebo_extras)

central = round(a2)
low = round(a3)
high = round(grid_extra - placebo_average)
print('central:', central, '  low:', low, '  high:', high)

central: 3499   low: 1755   high: 4842


In [104]:
campaign_bookings= pd.read_csv('campaign_bookings.csv')
visit_cost = 0

offer_count = bookings['in_offer'].sum()
fees_waived = campaign_bookings['waiver_rs'].sum()
new_home_visits = offer_count - table.loc['2025Q1', 'home']   # moved from clinic + extra
total_cost = fees_waived + visit_cost * new_home_visits

In [105]:
import math
print('fees waived:', fees_waived)
print('cost per extra booking (central):', round(total_cost / central))
print('cost per extra booking (range):', round(total_cost / high), 'to', round(total_cost / low))
print('break-even extra bookings:', math.ceil(total_cost / value))

fees waived: 7330350
cost per extra booking (central): 2095
cost per extra booking (range): 1514 to 4177
break-even extra bookings: 4527


In [106]:
print('double an offer that does not work, extra spend:', total_cost)
print('cut an offer that works, revenue lost (central):', round(central * value))
print('cut an offer that works, revenue lost (high):', round(high * value))

double an offer that does not work, extra spend: 7330350
cut an offer that works, revenue lost (central): 5665899
cut an offer that works, revenue lost (high): 7840607


In [107]:
share = round(central / offer_count * 100)
print('About', round(central, -2), 'of the', offer_count, 'free home bookings in January-March 2026',
      '(' + str(share) + '%) were bookings Kalpa would not otherwise have had.')

About 3500 of the 48869 free home bookings in January-March 2026 (7%) were bookings Kalpa would not otherwise have had.
